# 00 — Сборка датасета

## Источники данных

### Emotion-датасеты (точная разметка)
| Датасет | Размер | Источник | Классы |
|---|---|---|---|
| `seara/ru_go_emotions` | ~54k | Reddit (Google Translate → RU) | 28 → Ekman |
| `cedr` | ~9.4k | Блоги, новости, Twitter (нативный RU) | 5 → Ekman |
| `Djacon/ru-izard-emotions` | ~30k | Reddit (DeepL → RU) | 10 → Ekman |
| **BRIGHTER** (опц.) | ~тыс. | Толока (нативный RU) | 6 Ekman |

### Sentiment-датасеты (приблизительный маппинг: pos→joy, neu→neutral, neg→sadness)
| Датасет | Размер | Источник |
|---|---|---|
| RuReviews | 90k | Интернет-магазин (одежда) |
| RuSentiTweet | ~13k | Twitter (ручная разметка) |

> ⚠️ **BRIGHTER**: требует ручной загрузки с https://www.codabench.org/competitions/3863/
> После регистрации скачай архив, распакуй и укажи путь в ячейке конфигурации.

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(WORKING_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

## Конфигурация источников

Включи/выключи нужные датасеты:

In [ ]:
# ── Что включать ──────────────────────────────────────────────────
USE_RU_GO_EMOTIONS   = True   # seara/ru_go_emotions (~54k)
USE_CEDR             = True   # cedr (~9.4k, нативный RU)
USE_RU_IZARD         = True   # Djacon/ru-izard-emotions (~30k)
USE_RUREVIEWS        = True   # RuReviews (90k, sentiment→Ekman)
USE_RUSENTITWEET     = True   # RuSentiTweet (13k, sentiment→Ekman)

# BRIGHTER via HuggingFace (без ручной загрузки):
USE_BRIGHTER_HF = True   # brighter-dataset/BRIGHTER-emotion-categories (русский, ~тыс.)

# BRIGHTER через ручную загрузку: укажи путь или оставь None
BRIGHTER_DIR = None  # Например: '/kaggle/input/brighter-russian/'

# Dusha: ~300k нативных RU транскриптов речи (4 класса → Ekman)
USE_DUSHA         = True    # KELONMYOSA/dusha_emotion_audio
DUSHA_MAX_SAMPLES = 50_000  # max per split; None = без ограничения

# Максимум примеров на класс после балансировки (None = без ограничения)
MAX_PER_CLASS = 15_000
print('Конфигурация загружена')

## 1. Emotion-датасеты (HuggingFace)

In [ ]:
from src.data_loader import (
    load_ru_go_emotions, load_cedr, load_ru_izard_emotions,
    load_brighter, load_brighter_hf, load_dusha,
    load_rureviews, load_rusentitweet,
    merge_datasets, EKMAN_ID2LABEL, EKMAN_LABEL_NAMES,
)

emotion_sources = {}

if USE_RU_GO_EMOTIONS:
    try:
        emotion_sources['ru_go_emotions'] = load_ru_go_emotions(config='simplified')
    except Exception as e:
        print(f'ru_go_emotions failed: {e}')

if USE_CEDR:
    try:
        emotion_sources['cedr'] = load_cedr()
    except Exception as e:
        print(f'cedr failed: {e}')

if USE_RU_IZARD:
    try:
        emotion_sources['ru_izard'] = load_ru_izard_emotions()
    except Exception as e:
        print(f'ru_izard failed: {e}')

if USE_BRIGHTER_HF:
    try:
        emotion_sources['brighter_hf'] = load_brighter_hf()
    except Exception as e:
        print(f'BRIGHTER-HF failed: {e}')

if BRIGHTER_DIR and os.path.isdir(BRIGHTER_DIR):
    try:
        emotion_sources['brighter'] = load_brighter(BRIGHTER_DIR)
    except Exception as e:
        print(f'BRIGHTER (local) failed: {e}')
else:
    if not USE_BRIGHTER_HF:
        print('BRIGHTER: пропущен (BRIGHTER_DIR не указан и USE_BRIGHTER_HF=False)')

if USE_DUSHA:
    try:
        from src.data_loader import load_dusha
        emotion_sources['dusha'] = load_dusha(max_samples=DUSHA_MAX_SAMPLES)
    except Exception as e:
        print(f'Dusha failed: {e}')

print(f'\nЗагружено emotion-источников: {len(emotion_sources)}')

## 2. Sentiment-датасеты (приблизительный маппинг pos→joy, neg→sadness, neu→neutral)

In [ ]:
sentiment_sources = {}

if USE_RUREVIEWS:
    try:
        sentiment_sources['rureviews'] = load_rureviews()
    except Exception as e:
        print(f'RuReviews failed: {e}')

if USE_RUSENTITWEET:
    try:
        sentiment_sources['rusentitweet'] = load_rusentitweet()
    except Exception as e:
        print(f'RuSentiTweet failed: {e}')

print(f'Загружено sentiment-источников: {len(sentiment_sources)}')

In [ ]:
# Визуализируем распределение каждого источника
EMOTION_COLORS = {
    'anger': '#e74c3c', 'disgust': '#8e44ad', 'fear': '#2c3e50',
    'joy': '#f39c12', 'sadness': '#3498db', 'surprise': '#1abc9c', 'neutral': '#95a5a6',
}

all_sources = {**emotion_sources, **sentiment_sources}
n = len(all_sources)
if n == 0:
    print('Нет загруженных датасетов!')
else:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1: axes = [axes]
    for ax, (name, ds) in zip(axes, all_sources.items()):
        df = pd.concat([ds[s].to_pandas() for s in ds])
        counts = df['label'].map(EKMAN_ID2LABEL).value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0)
        ax.bar(counts.index, counts.values, color=[EMOTION_COLORS[e] for e in counts.index])
        ax.set_title(f'{name}\n({len(df):,})')
        ax.tick_params(axis='x', rotation=40)
    plt.suptitle('Распределение по источникам', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/per_source_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Объединение и балансировка

In [ ]:
all_ds_to_merge = {**emotion_sources, **sentiment_sources}
if not all_ds_to_merge:
    raise RuntimeError('Нет данных для объединения!')

dataset_raw = merge_datasets(all_ds_to_merge, test_size=0.15, val_size=0.15, seed=42)

In [ ]:
# Балансировка: ограничиваем максимум на класс в train
from datasets import Dataset, DatasetDict
from sklearn.utils import resample

train_df = dataset_raw['train'].to_pandas()
print('Train до балансировки:')
print(train_df['label'].map(EKMAN_ID2LABEL).value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0).astype(int).to_string())

if MAX_PER_CLASS is not None:
    parts = []
    for lid in range(len(EKMAN_LABEL_NAMES)):
        sub = train_df[train_df['label'] == lid]
        if len(sub) == 0:
            print(f'  WARNING: нет примеров для {EKMAN_ID2LABEL[lid]}')
            continue
        if len(sub) > MAX_PER_CLASS:
            sub = resample(sub, n_samples=MAX_PER_CLASS, random_state=42, replace=False)
        parts.append(sub)
    train_df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print('\nTrain после балансировки:')
print(train_df['label'].map(EKMAN_ID2LABEL).value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0).astype(int).to_string())

dataset_balanced = DatasetDict({
    'train':      Dataset.from_pandas(train_df),
    'validation': dataset_raw['validation'],
    'test':       dataset_raw['test'],
})

In [ ]:
# Финальная визуализация
all_df = pd.concat([dataset_balanced[s].to_pandas() for s in dataset_balanced])
all_df['emotion'] = all_df['label'].map(EKMAN_ID2LABEL)

counts = all_df['emotion'].value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0)
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(counts.index, counts.values, color=[EMOTION_COLORS[e] for e in counts.index], edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(v):,}\n({v/len(all_df)*100:.1f}%)', ha='center', fontsize=9)
ax.set_title(f'Финальный датасет (всего {len(all_df):,})')
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/final_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nИтого по сплитам:')
for split in dataset_balanced:
    print(f'  {split:12s}: {len(dataset_balanced[split]):,}')

## 4. Предобработка и сохранение

In [ ]:
from src.preprocessor import preprocess_batch

print('Очищаю тексты...')
dataset_clean = dataset_balanced.map(
    lambda batch: preprocess_batch(batch, text_column='text', clean=True),
    batched=True, batch_size=1000, desc='Cleaning',
)
print('Готово!')

In [ ]:
# Сохранение
DATA_SAVE_PATH = f'{WORKING_DIR}/processed_data'
dataset_clean.save_to_disk(DATA_SAVE_PATH)

with open(f'{WORKING_DIR}/label_names.json', 'w') as f:
    json.dump(EKMAN_LABEL_NAMES, f)

# Метаданные о составе датасета
meta = {
    'sources': {
        'emotion': list(emotion_sources.keys()),
        'sentiment_approx': list(sentiment_sources.keys()),
        'brighter': BRIGHTER_DIR is not None,
    },
    'max_per_class': MAX_PER_CLASS,
    'splits': {split: len(dataset_clean[split]) for split in dataset_clean},
    'label_names': EKMAN_LABEL_NAMES,
}
with open(f'{WORKING_DIR}/dataset_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

for split in ['train', 'validation', 'test']:
    df = dataset_clean[split].to_pandas()
    if 'label' in df.columns:
        df['emotion'] = df['label'].map(EKMAN_ID2LABEL)
    df.to_csv(f'{WORKING_DIR}/{split}.csv', index=False)
    print(f'{split}.csv → {len(df):,} строк')

print(f'\nДатасет сохранён в: {WORKING_DIR}')